# Pairs Trading: A Statistical Arbitrage Strategy Based on Cointegration and Z-Score

**Author:** Alexis Gardy, ESILV, MSc in Financial Engineering
**Stack:** Python · pandas · NumPy · statsmodels · matplotlib

---

## Overview

Pairs trading is one of the oldest **statistical arbitrage** strategies. It was developed in the mid-1980s by Nunzio Tartaglia's quantitative group at Morgan Stanley, and later formalized in the academic literature by Gatev, Goetzmann and Rouwenhorst (*Pairs Trading: Performance of a Relative-Value Arbitrage Rule*, 2006). Rather than forecasting the direction of the market, it trades the **relative value** of two assets.

### Economic intuition
Two companies operating in the same industry with similar business models (Coca-Cola / PepsiCo, Visa / Mastercard, ExxonMobil / Chevron…) share common risk factors. In the short run their prices may drift apart because of order flows, idiosyncratic news or noise. Over the long run, economic forces tend to pull them back toward an equilibrium relationship.

The strategy exploits these temporary deviations. When the **spread** between the two assets moves abnormally far from its mean, we go **long the relatively undervalued asset** and **short the relatively overvalued one**, then unwind the position once the spread reverts. Because the two legs offset each other, market exposure is largely neutralized: performance depends on the behavior of the spread, not on the overall market trend.

### Research question
> *Can we identify a statistically robust equilibrium relationship between two assets, and exploit it with systematic trading rules that remain profitable out of sample, net of transaction costs?*

### Methodology
1. **Data**: collection, cleaning and alignment of adjusted price series
2. **Exploratory analysis**: returns, correlation, and why correlation is not enough
3. **Train / test split**: separating calibration and evaluation periods
4. **Cointegration**: stationarity tests (ADF), Engle-Granger procedure, hedge ratio estimation
5. **Spread dynamics**: Ornstein-Uhlenbeck model and mean-reversion half-life
6. **Signal**: rolling z-score of the spread
7. **Trading rules**: entry, exit and stop-loss thresholds
8. **Backtest**: execution lag, transaction costs
9. **Performance evaluation**: risk-adjusted metrics
10. **Robustness**: parameter sensitivity and out-of-sample validation

### Assumptions and limitations
- All parameters are estimated **on the training period only**, to avoid look-ahead bias.
- The backtest assumes execution at the close, unrestricted short selling and simple proportional transaction costs. It does not model market impact or securities lending fees.
- The main risk is a **breakdown of the cointegration relationship** (change in business model, merger, sector-specific shock).

### Glossary
| Term | Definition |
|---|---|
| **Spread** | Linear combination of the two log-prices, $s_t = \ln A_t - \beta \ln B_t$ |
| **Hedge ratio β** | Amount of B sold short for each unit of capital invested in A |
| **Cointegration** | Two non-stationary series are cointegrated if a linear combination of them is stationary |
| **Z-score** | Deviation of the spread from its rolling mean, in number of standard deviations |
| **Half-life** | Expected time for a deviation of the spread to shrink by half |

---
## 1. Setup

In [1]:
import pandas as pd 
import numpy as np 
import statsmodels.api as sm
import plotly.graph_objects as go
import yfinance as yf 
import plotly.io as pio


pio.templates.default = "plotly_white"
pio.renderers.default = "notebook+png"
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)

# Set a random seed for reproducibility
rng = np.random.default_rng(42)

---
## 2. Data Collection

A good pair consists of two companies in the **same industry**, with **comparable business models** and exposure to the same economic drivers. Candidate pairs include `KO`/`PEP`, `XOM`/`CVX`, `V`/`MA`, `HD`/`LOW`, `GS`/`MS` or the ETFs `EWA`/`EWC`.

Daily data covering several years is required. **Adjusted close prices** should be preferred, since they account for dividends and stock splits.

In [6]:
# Pair selection: two US beverage companies with similar business models
TICKER_A = "KO"    # Coca-Cola
TICKER_B = "PEP"   # PepsiCo

START_DATE = "2015-01-01"
END_DATE = "2025-12-31"

# Download daily OHLCV data (prices adjusted for splits and dividends)
df_a = yf.download(TICKER_A, start=START_DATE, end=END_DATE,
                   auto_adjust=True, multi_level_index=False, progress=False)
df_b = yf.download(TICKER_B, start=START_DATE, end=END_DATE,
                   auto_adjust=True, multi_level_index=False, progress=False)

# Save raw data locally so the analysis remains reproducible
df_a.to_csv(f"data/{TICKER_A}.csv")
df_b.to_csv(f"data/{TICKER_B}.csv")

# Inspect structure
for ticker, df in [(TICKER_A, df_a), (TICKER_B, df_b)]:
    print(f"===== {ticker} =====")
    print(f"Rows: {len(df)} | Columns: {list(df.columns)}")
    print(f"Index type: {type(df.index).__name__} | Period: {df.index.min().date()} -> {df.index.max().date()}")
    print(f"Missing values:\n{df.isna().sum().to_string()}\n")
    display(df.head())

===== KO =====
Rows: 2765 | Columns: ['Close', 'High', 'Low', 'Open', 'Volume']
Index type: DatetimeIndex | Period: 2015-01-02 -> 2025-12-30
Missing values:
Close     0
High      0
Low       0
Open      0
Volume    0



,Close,High,Low,Open,Volume
Date,,,,,
2015-01-02,29.215918,29.396179,28.980193,29.299114,9921100
2015-01-05,29.215918,29.791363,29.174321,29.597235,26292600
2015-01-06,29.437784,29.770571,29.285259,29.403119,16897500
2015-01-07,29.805254,29.888450,29.520998,29.673524,13412300
2015-01-08,30.165749,30.207348,29.881493,29.936959,21743600


===== PEP =====
Rows: 2765 | Columns: ['Close', 'High', 'Low', 'Open', 'Volume']
Index type: DatetimeIndex | Period: 2015-01-02 -> 2025-12-30
Missing values:
Close     0
High      0
Low       0
Open      0
Volume    0



,Close,High,Low,Open,Volume
Date,,,,,
2015-01-02,65.879158,66.506973,65.607103,66.220969,3545700
2015-01-05,65.383881,66.304682,65.376903,65.886131,6441000
2015-01-06,64.888573,66.004700,64.832771,65.628008,6195000
2015-01-07,66.786011,67.127827,65.872187,66.360488,6526300
2015-01-08,67.999794,68.299752,67.441730,67.483583,7131600


---
## 3. Data Cleaning and Alignment

The two raw datasets must be merged into a single, consistent time series of prices: one column per asset, indexed by date, sorted chronologically, restricted to the trading days common to both assets, with no missing values.

In [7]:
# TODO: select the relevant price column for each asset
df_a_price = df_a["Close"]
df_b_price = df_b["Close"]
df_a = df_a_price.to_frame(name=TICKER_A)
df_b = df_b_price.to_frame(name=TICKER_B)
# TODO: merge both series on common dates
df = pd.merge(df_a, df_b, left_index=True, right_index=True)
# TODO: handle missing values and duplicates
df = df.dropna()
df = df.drop_duplicates()
# TODO: sanity-check the resulting dataset
print(f"Final dataset shape: {df.shape}")
print (df.head())

Final dataset shape: (2764, 2)
                   KO        PEP
Date                            
2015-01-02  29.215918  65.879158
2015-01-05  29.215918  65.383881
2015-01-06  29.437784  64.888573
2015-01-07  29.805254  66.786011
2015-01-08  30.165749  67.999794


---
## 4. Exploratory Analysis

Before any statistical modelling, we visually inspect how the two assets have evolved. Since their price levels differ, they are also compared on a normalized basis.

In [ ]:
# TODO: plot the raw price series
# TODO: plot both series rebased to 100 at the start date
# TODO: comment on how closely the two assets move together over time

---
## 5. Returns and Correlation

Correlation is the most intuitive measure of co-movement, but it must be applied with care. Correlating **price levels** of non-stationary series can produce high values purely by chance (*spurious regression*, Granger & Newbold, 1974). Correlation of **returns** measures day-to-day co-movement, but says nothing about whether the price gap between two assets will close.

This section quantifies both and illustrates the limits of correlation as a pair selection criterion.

In [ ]:
# TODO: compute log-prices and daily log-returns
# TODO: compute the correlation of price levels and the correlation of returns
# TODO: compute and plot a rolling correlation of returns

In [ ]:
# TODO: simulate pairs of independent random walks and study the distribution of their price correlation
# TODO: conclude on the relevance of price correlation for pair selection

---
## 6. Train / Test Split

To produce an honest evaluation, every quantity estimated from the data (hedge ratio, half-life, trading parameters) must be calibrated on a **training period** and then assessed on a separate **test period** the model has never seen. Since we are working with time series, the split must respect chronological order.

In [ ]:
# TODO: split the data chronologically into a training set and a test set

---
## 7. Stationarity and Cointegration

### 7.1 Stationarity
A stationary series has constant statistical properties over time and tends to revert to its mean. The **Augmented Dickey-Fuller (ADF)** test checks the null hypothesis of a unit root (non-stationarity). Asset prices are typically non-stationary, i.e. integrated of order one, I(1), while their returns are stationary.

### 7.2 Cointegration (Engle-Granger)
Two I(1) series are **cointegrated** if there exists β such that

$$ \varepsilon_t = \ln A_t - \alpha - \beta \ln B_t $$

is stationary. The two prices may wander, but they wander **together**. The two-step Engle-Granger procedure estimates this relationship by OLS, then tests the residuals for stationarity. Because the residuals are estimated, specific critical values (MacKinnon) apply.

In [ ]:
# TODO: run ADF tests on log-prices and on returns for both assets (training period)

In [ ]:
# TODO: estimate the cointegrating relationship by OLS on the training period (alpha, beta)
# TODO: test the residuals for stationarity / run the Engle-Granger cointegration test
# TODO: plot the residuals and interpret the results

---
## 8. Spread and Mean-Reversion Half-Life

Using the hedge ratio estimated on the training set, the spread is defined over the full sample as

$$ s_t = \ln A_t - \beta \ln B_t $$

Its mean-reverting behavior is modelled as a discretized **Ornstein-Uhlenbeck** process:

$$ \Delta s_t = \theta \, s_{t-1} + c + \eta_t, \qquad \text{half-life} = -\frac{\ln 2}{\theta} $$

The half-life gives the typical holding period of a trade and is a natural reference for calibrating the signal.

In [ ]:
# TODO: build the spread over the full sample using the training hedge ratio
# TODO: estimate theta and the half-life on the training period
# TODO: plot the spread and mark the train / test boundary

---
## 9. Trading Signal: Rolling Z-Score

The raw spread has no intuitive scale. It is standardized into a z-score

$$ z_t = \frac{s_t - \mu_t}{\sigma_t} $$

where $\mu_t$ and $\sigma_t$ are the **rolling** mean and standard deviation of the spread over a lookback window, using only information available at time $t$.

In [ ]:
# TODO: choose a lookback window and justify the choice
# TODO: compute the rolling z-score of the spread
# TODO: plot the z-score together with the trading thresholds

---
## 10. Trading Rules

Positions are taken on the spread as a whole:
- **Long spread (+1)**: long A, short β·B, when the spread is abnormally low
- **Short spread (−1)**: short A, long β·B, when the spread is abnormally high
- **Flat (0)**: no position

| Condition | Action |
|---|---|
| Flat and $z_t < -z_{entry}$ | Open long spread |
| Flat and $z_t > +z_{entry}$ | Open short spread |
| Position open and $\lvert z_t \rvert < z_{exit}$ | Close position (mean reversion) |
| Position open and $\lvert z_t \rvert > z_{stop}$ | Close position (stop-loss) |

In [ ]:
# TODO: implement a function that converts the z-score into a series of positions (-1, 0, +1)
# TODO: test it on simple hand-made z-score sequences
# TODO: generate and plot the positions for the chosen parameters

---
## 11. Backtest

The daily return of a position on the spread combines the returns of both legs, weighted by the hedge ratio and expressed per unit of gross capital. Two classic pitfalls must be handled explicitly:
- **Look-ahead bias**: a signal computed at the close of day $t$ can only be traded from day $t+1$.
- **Transaction costs**: every change in position incurs costs (bid-ask spread, fees, borrowing).

In [ ]:
# TODO: compute the daily return of the spread portfolio
# TODO: apply positions with the appropriate execution lag
# TODO: deduct transaction costs on each position change
# TODO: build and plot the equity curve, separating train and test periods

---
## 12. Performance Evaluation

The strategy is assessed with standard risk-adjusted metrics, reported separately for the training and test periods: total and annualized return, annualized volatility, Sharpe ratio, maximum drawdown, number of trades and market exposure.

In [ ]:
# TODO: implement the performance metrics
# TODO: report them in a train vs. test summary table
# TODO: interpret the differences between the two periods

---
## 13. Robustness and Out-of-Sample Validation

The choice of lookback window and thresholds strongly affects results. Optimizing them carelessly leads to **overfitting**: with enough combinations, some will look profitable by chance. Parameters are therefore selected on the training period only, favoring stable regions of the parameter space, and evaluated **once** on the test period.

In [ ]:
# TODO: evaluate the strategy over a grid of parameters on the training period
# TODO: visualize the results (e.g. heatmap of Sharpe ratios)
# TODO: select the final parameters and evaluate them once on the test period

---
## 14. Conclusion

*Summarize the main findings: strength of the cointegration relationship, out-of-sample performance net of costs, key risks and limitations.*

---
## 15. Further Work

- Time-varying hedge ratio (rolling OLS, Kalman filter)
- Rolling cointegration tests to detect relationship breakdowns
- Systematic pair selection across a sector, accounting for multiple testing
- Johansen test for baskets of three or more assets
- More realistic costs: securities lending fees, market impact
- Walk-forward optimization

## References
- Gatev, E., Goetzmann, W. N., & Rouwenhorst, K. G. (2006). *Pairs Trading: Performance of a Relative-Value Arbitrage Rule*. Review of Financial Studies, 19(3), 797–827.
- Engle, R. F., & Granger, C. W. J. (1987). *Co-integration and Error Correction: Representation, Estimation, and Testing*. Econometrica, 55(2), 251–276.
- Granger, C. W. J., & Newbold, P. (1974). *Spurious Regressions in Econometrics*. Journal of Econometrics, 2(2), 111–120.
- Chan, E. (2013). *Algorithmic Trading: Winning Strategies and Their Rationale*. Wiley.
- Vidyamurthy, G. (2004). *Pairs Trading: Quantitative Methods and Analysis*. Wiley.